In [1]:
import pandas as pd
import numpy as np


In [2]:
#uvoz podataka
#sezona 23/24
df_23_4 = pd.read_csv("super-lig-2023.csv")
#sezona 24/25
df_24_5 = pd.read_csv("super-lig-2024.csv")
#sezona 25/6
df_25_6 = pd.read_csv("super-lig-2025.csv")

## Obavezno!!!

In [3]:
#odrediti duzinu sezone
df_25_6 = df_25_6.head(126)
df_25_6.tail()

,Match Number,Round Number,Date,Location,Home Team,Away Team,Result
121,120,14,29/11/2025 20:00,Papara Park,Trabzonspor,Konyaspor,3 - 1
122,119,14,30/11/2025 17:00,Antalya Stadium,Antalyaspor,Göztepe,1 - 2
123,122,14,30/11/2025 20:00,Atatürk Olympic Stadium,Fatih Karagümrük,Besiktas,0 - 2
124,121,14,01/12/2025 20:00,Samsun 19 Mayis Stadium,Samsunspor,Alanyaspor,1 - 1
125,124,14,01/12/2025 20:00,Ülker Stadium,Fenerbahçe,Galatasaray,1 - 1


## Priprema Podataka i df-a

In [4]:
#sjedinjavanje ta tri df niza podataka - po osi jedan - odnosno uzduž
df_BL = pd.concat([df_23_4, df_24_5, df_25_6], axis=0)
df_BL = df_BL.drop(["Match Number","Round Number","Date","Location"],axis=1)
df_BL.rename(columns={"Home Team":"H","Away Team":"A"},inplace=True)

rez_dom = df_BL["Result"].str.slice(0,1)
rez_gost = df_BL["Result"].str.slice(4,5)

df_rezultatski = pd.concat([df_BL,rez_dom,rez_gost],axis=1)
df_rezultatski.columns = ["H","A","Result","Rez1","Rez2"]

df1 = df_rezultatski

## Definisanje Funckije za Izracun Forme

In [5]:
def klub(tim):
    # Filtriramo DataFrame
    df = df1[(df1["H"] == tim) | (df1["A"] == tim)].copy()
    
    uslovi = [
    (df['H'] == tim) & (df['Rez1'] < df["Rez2"]),
    (df['A'] == tim) & (df['Rez1'] < df["Rez2"]),
    (df['H'] == tim) & (df['Rez1'] == df["Rez2"]),
    (df['A'] == tim) & (df['Rez1'] == df["Rez2"]),
    (df['H'] == tim) & (df['Rez1'] > df["Rez2"]),
    (df['A'] == tim) & (df['Rez1'] > df["Rez2"]),]
    
    vrijednosti = [-1, 1, 0, 0, 1, -1]
    df['Forma'] = np.select(uslovi, vrijednosti, default=0)
    df = df.drop(["Rez1","Rez2"],axis=1).tail(5)
    rezultat = df["Forma"].sum() / len(df["Forma"])
    return rezultat

## Provjera Rada te Funckije

In [6]:
klub('Kasimpasa')

-0.4

## Stvaranje Petlje za Automatski Izracun Forme Svih Klubova

In [7]:
#izvlacenje naziva svih klubova
timovi = pd.concat([df1["H"], df1["A"]]).unique()
timovi

array(['Trabzonspor', 'Kasimpasa', 'Konyaspor', 'Kayserispor',
       'Pendikspor', 'Sivasspor', 'Adana Demirspor', 'Fenerbahçe',
       'Alanyaspor', 'Fatih Karagümrük', 'Istanbulspor', 'Antalyaspor',
       'Çaykur Rizespor', 'Hatayspor', 'Galatasaray',
       'Istanbul Basaksehir', 'Gaziantep', 'Besiktas', 'Samsunspor',
       'Ankaragücü', 'Bodrum', 'Göztepe', 'Eyüpspor', 'Kocaelispor',
       'Gençlerbirligi'], dtype=object)

In [8]:
rezultati = {}

In [9]:
#petlja
for tim in timovi:
    rezultat = klub(tim)
    rezultati[tim] = rezultat

# Kreiranje DataFrame-a iz rezultata
df_forme = pd.DataFrame(list(rezultati.items()),columns = ["Klub","Rezultat"])
df_forme["Rezultat"] = df_forme["Rezultat"]

## Rezulat Forme za Sve

In [10]:
#Izracun
df_forme 

,Klub,Rezultat
0,Trabzonspor,0.6
1,Kasimpasa,-0.4
2,Konyaspor,-0.4
3,Kayserispor,0.0
4,Pendikspor,0.0
5,Sivasspor,-0.4
6,Adana Demirspor,-0.4
7,Fenerbahçe,0.8
8,Alanyaspor,-0.4
9,Fatih Karagümrük,-0.2


In [11]:
#Ponovna provjera vrijednosti...
klub("Kasimpasa")

-0.4

In [12]:
df_forme.to_csv("forma_h.csv")

In [13]:
df_forme["Rezultat"] = df_forme["Rezultat"] * -1

In [14]:
df_forme


,Klub,Rezultat
0,Trabzonspor,-0.6
1,Kasimpasa,0.4
2,Konyaspor,0.4
3,Kayserispor,-0.0
4,Pendikspor,-0.0
5,Sivasspor,0.4
6,Adana Demirspor,0.4
7,Fenerbahçe,-0.8
8,Alanyaspor,0.4
9,Fatih Karagümrük,0.2


In [15]:
df_forme.to_csv("forma_a.csv")